## 1. Setup and Imports

In [ ]:
# Install required packages
%pip install neqsim pandas matplotlib numpy scipy -q

# Add parent path for local package
import sys
sys.path.insert(0, '../../../python')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# NeqSim imports
from neqsim.thermo.thermoTools import fluid, TPflash

# Local utilities
from neqsim_fitting.data import load_vle_data, get_data_summary
from neqsim_fitting.evaluation import calculate_all_metrics, parity_plot, deviation_plot
from neqsim_fitting.fitting import VLEFitter
from neqsim_fitting.reporting import generate_fitting_report, save_report

In [ ]:
# Configuration
COMPONENT_1 = 'CO2'  # Solute
COMPONENT_2 = 'water'  # Solvent
EQUATION_OF_STATE = 'cpa'
DATA_FILE = '../../../data/thermodynamics/VLE/water-CO2.csv'

## 2. Load Experimental Data

In [ ]:
# Load experimental data
expdata_df = pd.read_csv(DATA_FILE)
print(f"Loaded {len(expdata_df)} data points")
expdata_df.head(10)

In [ ]:
# Data summary
summary = get_data_summary(expdata_df)
print(f"Temperature range: {summary.get('temperature_range', 'N/A')} K")
print(f"Pressure range: {summary.get('pressure_range', 'N/A')} bar")
print(f"References: {summary.get('references', [])}")
print(f"Number of references: {summary.get('n_references', 0)}")

## 3. Data Visualization

In [ ]:
# Plot experimental data by temperature
fig, ax = plt.subplots(figsize=(10, 6))

temperatures = expdata_df['Temperature'].unique()
colors = plt.cm.viridis(np.linspace(0, 1, len(temperatures)))

for temp, color in zip(sorted(temperatures), colors):
    data_temp = expdata_df[expdata_df['Temperature'] == temp]
    ax.scatter(data_temp['x1'], data_temp['Pressure'], 
               c=[color], label=f'{temp:.1f} K', alpha=0.7)

ax.set_xlabel('x (mole fraction)')
ax.set_ylabel('Pressure (bar)')
ax.set_title(f'{COMPONENT_1}-{COMPONENT_2} VLE Data')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Data Quality Evaluation

Review the data quality and remove any problematic data points or references.

In [ ]:
# Check data by reference
ref_summary = expdata_df.groupby('Reference').agg({
    'Temperature': ['min', 'max', 'count'],
    'Pressure': ['min', 'max']
})
ref_summary

In [ ]:
# Filter data if needed (example: exclude specific references)
EXCLUDE_REFERENCES = []  # Add references to exclude, e.g., ['Nighswander1989']

if EXCLUDE_REFERENCES:
    expdata_filtered = expdata_df[~expdata_df['Reference'].isin(EXCLUDE_REFERENCES)]
    print(f"Removed {len(expdata_df) - len(expdata_filtered)} points")
else:
    expdata_filtered = expdata_df.copy()

print(f"Using {len(expdata_filtered)} data points for fitting")

## 5. Baseline Model Comparison

Compare experimental data with the default NeqSim model (without parameter adjustment).

In [ ]:
# Create fluid with default parameters
fluid1 = fluid(EQUATION_OF_STATE)
fluid1.addComponent(COMPONENT_1, 1.0)
fluid1.addComponent(COMPONENT_2, 1.0)
fluid1.autoSelectMixingRule()

# Get current kij parameters
phase = fluid1.getPhase(0)
mixing_rule = phase.getMixingRule()
idx1 = phase.getComponent(COMPONENT_1).getComponentNumber()
idx2 = phase.getComponent(COMPONENT_2).getComponentNumber()

current_kij = mixing_rule.getBinaryInteractionParameter(idx1, idx2)
current_kij_T = mixing_rule.getBinaryInteractionParameterT1(idx1, idx2)

print(f"Current kij: {current_kij}")
print(f"Current kij_T: {current_kij_T}")

In [ ]:
# Calculate baseline predictions
def calculate_solubility(temp, pres):
    """Calculate solubility at given T, P."""
    try:
        fluid1.setTemperature(temp, 'K')
        fluid1.setPressure(pres, 'bara')
        TPflash(fluid1)
        if fluid1.hasPhaseType('aqueous'):
            return fluid1.getPhase('aqueous').getComponent(COMPONENT_1).getx()
        return float('nan')
    except:
        return float('nan')

# Calculate for all data points
baseline_calc = [
    calculate_solubility(row['Temperature'], row['Pressure'])
    for _, row in expdata_filtered.iterrows()
]

expdata_filtered = expdata_filtered.assign(x_baseline=baseline_calc)

In [ ]:
# Baseline metrics
baseline_metrics = calculate_all_metrics(
    expdata_filtered['x1'].values,
    expdata_filtered['x_baseline'].values,
    name='Baseline'
)
print("Baseline Model Performance:")
print(f"  AAD%: {baseline_metrics['aad_percent']:.2f}%")
print(f"  RMSE: {baseline_metrics['rmse']:.6f}")
print(f"  Bias%: {baseline_metrics['bias_percent']:.2f}%")

## 6. Parameter Fitting

In [ ]:
# Initialize fitter
fitter = VLEFitter(
    component_1=COMPONENT_1,
    component_2=COMPONENT_2,
    equation_of_state=EQUATION_OF_STATE,
)

# Fit kij (and optionally kij_T)
result = fitter.fit_kij(
    expdata_filtered,
    fit_kij_T=True,  # Set to True to also fit temperature dependence
    initial_kij=current_kij,
    initial_kij_T=current_kij_T,
    kij_bounds=(-0.5, 0.5),
    kij_T_bounds=(-0.005, 0.005),
)

print(result.summary())

## 7. Validation and Error Analysis

In [ ]:
# Parity plot
parity_plot(
    expdata_filtered['x1'].values,
    result.calculated_values,
    title=f'{COMPONENT_1}-{COMPONENT_2} Parity Plot (Fitted)',
    xlabel='Experimental x',
    ylabel='Calculated x',
    deviation_lines=[5, 10, 20],
)

In [ ]:
# Deviation vs Temperature
deviation_plot(
    expdata_filtered['Temperature'].values,
    expdata_filtered['x1'].values,
    result.calculated_values,
    xlabel='Temperature (K)',
    title='Deviation vs Temperature',
)

In [ ]:
# Deviation vs Pressure
deviation_plot(
    expdata_filtered['Pressure'].values,
    expdata_filtered['x1'].values,
    result.calculated_values,
    xlabel='Pressure (bar)',
    title='Deviation vs Pressure',
)

In [ ]:
# Compare baseline vs fitted
print("\nComparison: Baseline vs Fitted")
print("="*50)
print(f"{'Metric':<20} {'Baseline':>12} {'Fitted':>12}")
print("-"*50)
print(f"{'AAD%':<20} {baseline_metrics['aad_percent']:>12.2f} {result.metrics['aad_percent']:>12.2f}")
print(f"{'RMSE':<20} {baseline_metrics['rmse']:>12.6f} {result.metrics['rmse']:>12.6f}")
print(f"{'Max Error%':<20} {baseline_metrics['max_error_percent']:>12.2f} {result.metrics['max_error_percent']:>12.2f}")

## 8. Export Results

In [ ]:
# Generate fitting report
report = generate_fitting_report(
    result,
    property_type='VLE Solubility',
    components=[COMPONENT_1, COMPONENT_2],
    equation_of_state=EQUATION_OF_STATE,
    data_references=expdata_filtered['Reference'].unique().tolist(),
    author='Your Name',
)

# Display report preview
from IPython.display import Markdown
display(Markdown(report))

In [ ]:
# Save report
from datetime import datetime
date_str = datetime.now().strftime('%Y-%m-%d')
report_path = f'../../../reports/generated/{date_str}-{COMPONENT_1}-{COMPONENT_2}-kij.md'
save_report(report, report_path)
print(f"Report saved to: {report_path}")

In [ ]:
# Export fitted data with deviations
export_df = expdata_filtered.copy()
export_df['x_fitted'] = result.calculated_values
export_df['deviation_percent'] = ((result.calculated_values - expdata_filtered['x1']) / expdata_filtered['x1']) * 100

export_path = f'./{COMPONENT_2}-{COMPONENT_1}-exp-fitted.csv'
export_df.to_csv(export_path, index=False)
print(f"Fitted data exported to: {export_path}")

## 9. Summary and Recommendations

### Fitted Parameters
| Parameter | Value |
|-----------|-------|
| kij | [Value] |
| kij_T | [Value] |

### Recommendations
- [Add your recommendations here]
- [Note any data quality issues]
- [Note temperature/pressure limitations]